In [22]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print("Core libraries imported")

Core libraries imported


In [23]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
print("Sklearn modules imported")

Sklearn modules imported


In [24]:
df = pd.read_csv("/home/user/Downloads/water_potability.csv")
print("Dataset loaded successfully")
print(f"Shape: {df.shape}")

Dataset loaded successfully
Shape: (3276, 10)


In [ ]:
df.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [26]:
df.tail()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1
3272,7.808856,193.553212,17329.802160,8.061362,NaN,392.449580,19.903225,NaN,2.798243,1
3273,9.419510,175.762646,33155.578218,7.350233,NaN,432.044783,11.039070,69.845400,3.298875,1
3274,5.126763,230.603758,11983.869376,6.303357,NaN,402.883113,11.168946,77.488213,4.708658,1
3275,7.874671,195.102299,17404.177061,7.509306,NaN,327.459760,16.140368,78.698446,2.309149,1


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ph               2785 non-null   float64
 1   Hardness         3276 non-null   float64
 2   Solids           3276 non-null   float64
 3   Chloramines      3276 non-null   float64
 4   Sulfate          2495 non-null   float64
 5   Conductivity     3276 non-null   float64
 6   Organic_carbon   3276 non-null   float64
 7   Trihalomethanes  3114 non-null   float64
 8   Turbidity        3276 non-null   float64
 9   Potability       3276 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 256.1 KB


In [28]:
print("Columns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

Columns: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']

Data Types:
ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object


In [29]:
df.describe().round(3)

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
count,2785.000,3276.000,3276.000,3276.000,2495.000,3276.000,3276.000,3114.000,3276.000,3276.000
mean,7.081,196.369,22014.093,7.122,333.776,426.205,14.285,66.396,3.967,0.390
std,1.594,32.880,8768.571,1.583,41.417,80.824,3.308,16.175,0.780,0.488
min,0.000,47.432,320.943,0.352,129.000,181.484,2.200,0.738,1.450,0.000
25%,6.093,176.851,15666.690,6.127,307.699,365.734,12.066,55.845,3.440,0.000
50%,7.037,196.968,20927.834,7.130,333.074,421.885,14.218,66.622,3.955,0.000
75%,8.062,216.667,27332.762,8.115,359.950,481.792,16.558,77.337,4.500,1.000
max,14.000,323.124,61227.196,13.127,481.031,753.343,28.300,124.000,6.739,1.000


In [30]:
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing': missing, 'Pct (%)': pct}))

                 Missing  Pct (%)
ph                   491    14.99
Hardness               0     0.00
Solids                 0     0.00
Chloramines            0     0.00
Sulfate              781    23.84
Conductivity           0     0.00
Organic_carbon         0     0.00
Trihalomethanes      162     4.95
Turbidity              0     0.00
Potability             0     0.00


In [31]:
print("Target distribution:")
print(df['Potability'].value_counts())
print("\nClass balance (%):", (df['Potability'].value_counts(normalize=True)*100).round(2))

Target distribution:
Potability
0    1998
1    1278
Name: count, dtype: int64

Class balance (%): Potability
0    60.99
1    39.01
Name: proportion, dtype: float64


In [32]:
dup = df.duplicated().sum()
print(f"Duplicate rows: {dup}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after deduplication: {df.shape}")

Duplicate rows: 0
Shape after deduplication: (3276, 10)


In [33]:
feature_cols = df.columns.drop('Potability').tolist()
knn_imputer = KNNImputer(n_neighbors=5)
df[feature_cols] = knn_imputer.fit_transform(df[feature_cols])
print("KNN Imputation done. Missing values remaining:", df.isnull().sum().sum())

KNN Imputation done. Missing values remaining: 0


In [34]:
outlier_info = {}
for col in feature_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    outlier_info[col] = n_out
print(pd.Series(outlier_info, name='Outlier Count'))

ph                  79
Hardness            83
Solids              47
Chloramines         61
Sulfate            114
Conductivity        11
Organic_carbon      25
Trihalomethanes     49
Turbidity           19
Name: Outlier Count, dtype: int64


In [35]:
for col in feature_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)
print("Outliers capped. Sample stats after clipping:")
print(df[feature_cols].describe().round(3))

Outliers capped. Sample stats after clipping:
             ph  Hardness     Solids  Chloramines   Sulfate  Conductivity  \
count  3276.000  3276.000   3276.000     3276.000  3276.000      3276.000   
mean      7.078   196.392  21957.112        7.122   333.677       426.130   
std       1.437    32.017   8592.820        1.544    35.546        80.564   
min       3.545   117.125    320.943        3.146   248.423       191.648   
25%       6.187   176.851  15666.690        6.127   312.378       365.734   
50%       7.052   196.968  20927.834        7.130   333.268       421.885   
75%       7.949   216.667  27332.762        8.115   355.014       481.792   
max      10.591   276.393  44831.870       11.096   418.968       655.879   

       Organic_carbon  Trihalomethanes  Turbidity  
count        3276.000         3276.000   3276.000  
mean           14.283           66.427      3.967  
std             3.288           15.590      0.776  
min             5.328           25.758      1.849  


In [36]:
X = df[feature_cols]
y = df['Potability'].astype(int)
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3276, 9)
y shape: (3276,)


In [37]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
print("Scaling complete. Mean ~0, Std ~1:")
print(X_scaled_df.mean().round(3))

Scaling complete. Mean ~0, Std ~1:
ph                -0.0
Hardness          -0.0
Solids            -0.0
Chloramines       -0.0
Sulfate           -0.0
Conductivity      -0.0
Organic_carbon     0.0
Trihalomethanes   -0.0
Turbidity         -0.0
dtype: float64


In [38]:
print("Standard deviation after scaling:")
print(X_scaled_df.std().round(3))

Standard deviation after scaling:
ph                 1.0
Hardness           1.0
Solids             1.0
Chloramines        1.0
Sulfate            1.0
Conductivity       1.0
Organic_carbon     1.0
Trihalomethanes    1.0
Turbidity          1.0
dtype: float64


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (2620, 9), Test: (656, 9)


In [40]:
print("Train class distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(3))

Train class distribution:
Potability
0    0.61
1    0.39
Name: proportion, dtype: float64

Test class distribution:
Potability
0    0.61
1    0.39
Name: proportion, dtype: float64


In [41]:
X_train.to_csv('/tmp/X_train.csv', index=False)
X_test.to_csv('/tmp/X_test.csv', index=False)
y_train.to_csv('/tmp/y_train.csv', index=False)
y_test.to_csv('/tmp/y_test.csv', index=False)
df.to_csv('/tmp/df_clean.csv', index=False)
X_scaled_df.to_csv('/tmp/X_scaled.csv', index=False)
y.to_csv('/tmp/y.csv', index=False)
print("All files saved to /tmp/")

All files saved to /tmp/


In [42]:
print("=" * 40)
print("  PREPROCESSING SUMMARY")
print("=" * 40)
print(f"  Clean shape     : {df.shape}")
print(f"  Features        : {len(feature_cols)}")
print(f"  Train samples   : {X_train.shape[0]}")
print(f"  Test  samples   : {X_test.shape[0]}")
print(f"  Missing values  : {df.isnull().sum().sum()}")
print("=" * 40)

  PREPROCESSING SUMMARY
  Clean shape     : (3276, 10)
  Features        : 9
  Train samples   : 2620
  Test  samples   : 656
  Missing values  : 0
